# Telecom Customer Churn Prediction

**Goal:** predict whether a customer will cancel their subscription next month. The retention team can use these predictions to prioritize outreach and offers.

This notebook cleans the data, compares two models, evaluates the best one on unseen customers, explains key drivers, and creates a retention list.

## 1. Import libraries and load data

In Kaggle, add the Telco Customer Churn dataset to the notebook. The first path is for Kaggle; the fallback makes the same notebook work locally.

In [ ]:
from pathlib import Path
import glob, warnings, joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

matches = glob.glob("/kaggle/input/**/WA_Fn-UseC_-Telco-Customer-Churn.csv", recursive=True)
data_path = Path(matches[0]) if matches else Path("archive/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

## 2. Check data quality and churn balance

Churn is a binary classification target: **Yes** means the customer left and **No** means they stayed. We inspect missing values and the target balance because accuracy can be misleading when the classes are uneven.

In [ ]:
df.info()
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_values"))
churn_distribution = df["Churn"].value_counts(normalize=True).mul(100).round(2)
display(churn_distribution.to_frame("percentage"))
ax = sns.countplot(data=df, x="Churn")
ax.set_title("Customer Churn Distribution")
ax.bar_label(ax.containers[0])
plt.show()

## 3. Clean features and create training/test data

`TotalCharges` is text because some new customers have blank values; converting it to numeric turns blanks into missing values, which our pipeline handles. We drop `customerID` because an identifier should not be used to memorize customers.

A stratified split preserves the churn proportion in the training and held-out test sets.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
print(f"Training: {len(X_train):,}; Test: {len(X_test):,}")
print("Numeric features:", numeric_features)

## 4. Build leakage-safe pipelines and compare models

Preprocessing happens inside each pipeline, so imputation, scaling, and one-hot encoding are learned only from training data. We compare Logistic Regression (interpretable baseline) and Random Forest (can capture non-linear patterns).

We select the model with the best **ROC-AUC**, which evaluates ranking quality across every possible probability threshold.

In [ ]:
numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}
fitted_pipelines, results = {}, {}
for name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    results[name] = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Test ROC-AUC"]).sort_values("Test ROC-AUC", ascending=False)
results_df.style.format("{:.3f}")

## 5. Evaluate the best model on unseen customers

The classification report provides precision, recall, and F1-score. For customer retention, churn recall is particularly valuable: it is the share of truly churning customers that the team successfully identifies.

In [ ]:
best_model_name = results_df.index[0]
best_pipeline = fitted_pipelines[best_model_name]
test_probabilities = best_pipeline.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.50).astype(int)

print(f"Selected model: {best_model_name}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.3f}")
print(classification_report(y_test, test_predictions, target_names=["No Churn", "Churn"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, test_predictions), display_labels=["No Churn", "Churn"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1])
axes[0].set_title("Confusion Matrix")
axes[1].set_title("ROC Curve")
plt.tight_layout()
plt.show()

## 6. Explain model drivers and produce the outreach list

Feature importance shows which service, billing, contract, and customer attributes most influence prediction. The retention list uses a 0.60 probability threshold for high-priority outreach; change it to match the team's budget and offer cost.

In [ ]:
feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()
model = best_pipeline.named_steps["model"]

if best_model_name == "Logistic Regression":
    importance = pd.DataFrame({"feature": feature_names, "value": model.coef_[0]})
    top_features = importance.assign(abs_value=importance["value"].abs()).nlargest(15, "abs_value").sort_values("value")
    chart_title = "Top Effects on Churn (positive = higher risk)"
else:
    importance = pd.DataFrame({"feature": feature_names, "value": model.feature_importances_})
    top_features = importance.nlargest(15, "value").sort_values("value")
    chart_title = "Top Random Forest Feature Importances"

plt.figure(figsize=(10, 7))
plt.barh(top_features["feature"], top_features["value"], color="#2a9d8f")
plt.title(chart_title)
plt.show()

retention_list = X_test.copy()
retention_list["actual_churn"] = y_test.values
retention_list["churn_probability"] = test_probabilities
retention_list["priority_outreach"] = np.where(test_probabilities >= 0.60, "High", "Standard")
retention_list = retention_list.sort_values("churn_probability", ascending=False)
display(retention_list[["churn_probability", "priority_outreach", "actual_churn"]].head(10))

retention_list.to_csv("retention_priority_list.csv", index=False)
joblib.dump(best_pipeline, "telco_churn_model.joblib")
print("Saved retention_priority_list.csv and telco_churn_model.joblib")

## Business recommendations

1. Contact high-probability customers first.
2. Tailor offers using the drivers found above—for example, service support, contract, payment method, and tenure.
3. Measure whether outreach saves customers, then retrain periodically with newer customer data.
4. Choose the final probability threshold by balancing offer cost against customer lifetime value.